# 15 - Contraste de hipótesis
En este notebook contrastamos H1, H2 y H5 con la matriz de experimentos. Seguimos el criterio de la memoria:
- Una hipótesis se refuta en un caso cuando la F1 macro del conjunto de variables que se evalúa es inferior al 80 % de la del conjunto `completo` (`CLEARLY_INFERIOR_RATIO`), con el mismo modelo, la misma estrategia y el mismo perfil.
- La matriz con los hiperparámetros ajustados es la principal y la de los valores por defecto sirve de comprobación.
- Una conclusión es robusta si coincide en los cuatro modelos (el dummy es solo línea base y no se usa).

Cada hipótesis tiene 12 casos, 3 estrategias por 4 perfiles.

## Carga de los resultados

In [ ]:
import pandas as pd

from urban_accessibility_ml.config import MATRIX_MODELS, REPORTS_DIR
from urban_accessibility_ml.modeling.evaluate import cases_below, summarize_folds

CASE = ["strategy", "feature_set", "profile"]
MODELS = [m for m in MATRIX_MODELS if m != "dummy"]
# Conjunto que se evalúa en cada hipótesis (tab:comparaciones-hipotesis de la memoria)
HYPOTHESES = {"H1": "cartografico_aereo", "H2": "calle", "H5": "combinado"}

summary = {
    name: summarize_folds(pd.read_csv(REPORTS_DIR / "results" / f"experiment-matrix-{name}.csv"), CASE + ["model"])
    for name in ["tuned", "default"]
}
summary["tuned"].shape, summary["default"].shape

## Casos por debajo del 80 % de completo
Número de casos de 12 en los que cada hipótesis queda por debajo del 80 % de completo, por modelo y con cada matriz.
- H1 se evalúa con `cartografico_aereo`.
- H2 con `calle`.
- H5 con `combinado`.

In [ ]:
def count_below(summary):
    counts = pd.DataFrame({m: cases_below(summary, m)["feature_set"].value_counts() for m in MODELS})
    return counts.reindex(list(HYPOTHESES.values())).fillna(0).astype(int).set_axis(list(HYPOTHESES))


display(count_below(summary["tuned"]).rename_axis("ajustada"))
display(count_below(summary["default"]).rename_axis("por defecto"))

Observamos que H1, sobre cartográfico y aéreo:
- En la regresión logística, 9 de los 12 casos de la matriz ajustada quedan por debajo del 80 % de completo, mientras que sin ajustar son 4 de 12.
- En random_forest, XGBoost y LightGBM hay pocos casos por debajo, 3 o menos tanto en ajustada como por defecto.

H2, sobre datos del conjunto a pie de calle:
- En ajustada, la regresión logística tiene 3 casos de 12 por debajo del 80 %, así como 1 caso de LightGBM.
- Mientras que en por defecto, únicamente 1 caso de LightGBM está por debajo del 80 %.

En cuanto a H5, sobre ambos conjuntos de imágenes, se repiten los mismos valores que a pie de calle.
Como esta comprobación se ha hecho con el umbral del 50 %, recordemos que el conjunto de calle y combinado tenía las mismas dimensiones.

## Casos concretos

Vamos a comprobar todos los casos por cada una de las hipótesis.

La columna `por debajo en` indica si el caso específico está por debajo en las dos matrices (ajustada y por defecto), solo en la ajustada o solo en la de por defecto.

In [ ]:
def cases(hypothesis):
    feature_set = HYPOTHESES[hypothesis]
    parts = {}
    for model in MODELS:
        below = cases_below(summary["tuned"], model)[CASE].merge(
            cases_below(summary["default"], model)[CASE], on=CASE, how="outer", indicator="por debajo en"
        )
        below = below[below["feature_set"] == feature_set]
        below["por debajo en"] = below["por debajo en"].map(
            {"both": "ambas", "left_only": "solo ajustada", "right_only": "solo por defecto"}
        )
        keys = list(zip(below["strategy"], below["profile"]))
        for name, label in [("tuned", "ajustada"), ("default", "por defecto")]:
            data = summary[name][summary[name]["model"] == model]

            def f1(fs):
                rows = data[data["feature_set"] == fs].set_index(["strategy", "profile"])
                return rows["f1_macro_mean"].loc[keys].to_numpy()

            below[f"F1 macro ({label})"] = f1(feature_set)
            below[f"% de completo ({label})"] = 100 * f1(feature_set) / f1("completo")
        parts[model] = below[
            ["strategy", "profile", "F1 macro (ajustada)", "% de completo (ajustada)",
             "F1 macro (por defecto)", "% de completo (por defecto)", "por debajo en"]
        ]
    columns = pd.concat(parts, names=["modelo"]).reset_index(level="modelo").reset_index(drop=True)
    return columns.round({"F1 macro (ajustada)": 3, "F1 macro (por defecto)": 3, "% de completo (ajustada)": 1, "% de completo (por defecto)": 1})

### H1 (`cartografico_aereo`)


In [ ]:
cases("H1")


Se puede ver que más de la mitad de los casos son de regresión logística, concretamente:
- El perfil sensorial tiene la F1 macro tanto de ajustada como por defecto por debajo del 80 % del completo, en las estrategias cross_city y mixed; así como within_city en ajustada.
- La estrategia mixta solo está por encima del 80 % en por defecto en cognitiva y funcional. El resto está por debajo del 80 % en ambos casos.
- El perfil funcional está por debajo en ajustada en todas las estrategias.

En los modelos de árboles:
- El caso mixed con perfil cognitivo está por debajo del 80 % tanto en F1 macro ajustada como por defecto, en los tres modelos de árbol.
- random_forest tiene mixed global por debajo solo en el de por defecto.
- XGBoost tiene tanto cross_city sensorial como mixed global por debajo del 80 % en la ajustada.
- LightGBM tiene por debajo cross_city funcional tanto en ajustada como por defecto.


### H2 (`calle`)


In [ ]:
cases("H2")


Usando H2, del conjunto de calle, nos encontramos:
- En regresión logística, la F1 macro está por debajo del 80 % de completo, solo en ajustada de cross_city cognitiva y sensorial, y mixed funcional.
- En LightGBM hay un caso por debajo, cross_city global, en ambas matrices.

### H5 (`combinado`)


In [ ]:
cases("H5")


Coincide con los valores de H2, porque con el umbral del 50 %, las dimensiones de calle y combinado son las mismas.

## Robustez de cada caso

Vamos a ver, para cada hipótesis, qué ocurre en cada uno de sus 12 casos (estrategia y perfil) con los cuatro modelos. La regla de lectura es la siguiente:
- Una hipótesis se considera refutada en un caso cuando lo está en los cuatro modelos, es decir, cuando la F1 macro queda por debajo del 80 % de completo en todos ellos.
- Se considera no refutada en un caso si no lo está en ninguno.
- En el resto de casos, depende del modelo.

In [ ]:
def flags(summary, feature_set):
    """True si el caso (estrategia, perfil) queda por debajo del 80 %, por modelo."""
    idx = summary[summary["feature_set"] == feature_set].set_index(["strategy", "profile"]).index.unique()
    out = pd.DataFrame(False, index=idx, columns=MODELS)
    for m in MODELS:
        below = cases_below(summary, m)
        below = below[below["feature_set"] == feature_set]
        out.loc[list(zip(below["strategy"], below["profile"])), m] = True
    return out


def verdicts(f):
    n = f.sum(axis=1)
    return n.map(lambda k: "no refutada" if k == 0 else "refutada" if k == len(MODELS) else "depende del modelo")


SHORT = {"logistic_regression": "LR", "random_forest": "RF", "xgboost_balanced": "XGB", "lightgbm": "LGBM"}


def below_models(f):
    """Cuántos modelos dejan el caso por debajo y cuáles, por ejemplo "2 de 4 (LR, LGBM)"."""
    def label(row):
        names = [SHORT[m] for m in MODELS if row[m]]
        return f"{len(names)} de {len(MODELS)}" + (f" ({', '.join(names)})" if names else "")

    return f.apply(label, axis=1)


def robustness(hypothesis):
    feature_set = HYPOTHESES[hypothesis]
    columns = {}
    for name, label in [("tuned", "ajustada"), ("default", "por defecto")]:
        f = flags(summary[name], feature_set)
        columns[f"modelos por debajo ({label})"] = below_models(f)
        columns[f"veredicto ({label})"] = verdicts(f)
    table = pd.DataFrame(columns)
    display(table)
    display(table.filter(like="veredicto").apply(pd.Series.value_counts).fillna(0).astype(int))

### H1 (`cartografico_aereo`)


In [ ]:
robustness("H1")

### H2 (`calle`)


In [ ]:
robustness("H2")

### H5 (`combinado`)


In [ ]:
robustness("H5")